# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mujahid1hm/flyrank-ai-/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is a ranking/scoring problem. The decision is not simply whether a page is declining, but which pages a content editor should review first. A score or ranked list is more useful than a binary label because the business action is to work the highest-priority pages first.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

lane = "ranking/scoring"
decision = "Which pages should an editor refresh first?"
why = "The output must be a prioritized queue, not just a yes/no label."

print(f"Task type: {lane}")
print(f"Decision: {decision}")
print(f"Why: {why}")


Task type: ranking/scoring
Decision: Which pages should an editor refresh first?
Why: The output must be a prioritized queue, not just a yes/no label.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The target is `is_declining_label`: whether a page is in a declining state based on its observed trend direction. This is an observed outcome from the trailing historical data, not a product-created rule. We are predicting a real later state from earlier page signals, which is the correct way to frame the target for a decision-support model.


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from pathlib import Path
import pandas as pd

search_roots = [Path.cwd(), *Path.cwd().parents]

def find_dataset():
    for root in search_roots:
        candidate = root / "data" / "raw" / "content_refresh_anonymized.csv"
        if candidate.exists():
            return candidate
    for root in search_roots:
        matches = list(root.rglob("content_refresh_anonymized.csv"))
        if matches:
            return matches[0]
    return None

candidate = find_dataset()
if candidate is None:
    raise FileNotFoundError("Starter dataset not found under data/raw/")

df = pd.read_csv(candidate)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
label_rate = df["is_declining_label"].mean()
print("Observed target definition: is_declining_label = (trend_direction == 'down')")
print(f"Observed decline rate in the starter data: {label_rate:.3f} ({label_rate * 100:.1f}%)")
print("This is an observed outcome from historical trend direction, not a hand-written rule used as a feature.")


Observed target definition: is_declining_label = (trend_direction == 'down')
Observed decline rate in the starter data: 0.542 (54.2%)
This is an observed outcome from historical trend direction, not a hand-written rule used as a feature.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The most defensible metric is Precision@K, especially Precision@20 or Precision@50. For the top K pages recommended for refresh, the key question is: what fraction are truly declining? If the model can surface a high share of real declines early, the editor can act on a smaller, more focused review list.


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from pathlib import Path
import pandas as pd

search_roots = [Path.cwd(), *Path.cwd().parents]

def find_dataset():
    for root in search_roots:
        candidate = root / "data" / "raw" / "content_refresh_anonymized.csv"
        if candidate.exists():
            return candidate
    for root in search_roots:
        matches = list(root.rglob("content_refresh_anonymized.csv"))
        if matches:
            return matches[0]
    return None

candidate = find_dataset()
if candidate is None:
    raise FileNotFoundError("Starter dataset not found under data/raw/")

df = pd.read_csv(candidate)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
base_rate = df["is_declining_label"].mean()
print("Recommended metric: Precision@K")
print(f"Baseline declining rate in the dataset: {base_rate:.3f} ({base_rate * 100:.1f}%)")
print("A good model should improve upon this baseline in the top 20 or top 50 pages recommended for refresh.")


Recommended metric: Precision@K
Baseline declining rate in the dataset: 0.542 (54.2%)
A good model should improve upon this baseline in the top 20 or top 50 pages recommended for refresh.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row is one content item (one page) with historical measurements for a client context. In this dataset, the unit is page-level, not page-day or keyword-level: each row describes a content record and its trailing 90-day performance signals.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from pathlib import Path
import pandas as pd

search_roots = [Path.cwd(), *Path.cwd().parents]

def find_dataset():
    for root in search_roots:
        candidate = root / "data" / "raw" / "content_refresh_anonymized.csv"
        if candidate.exists():
            return candidate
    for root in search_roots:
        matches = list(root.rglob("content_refresh_anonymized.csv"))
        if matches:
            return matches[0]
    return None

candidate = find_dataset()
if candidate is None:
    raise FileNotFoundError("Starter dataset not found under data/raw/")

df = pd.read_csv(candidate)
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print("One row = one content item / page, with trailing-90-day performance signals.")
print(df.head(3).to_string(index=False))


Rows: 30,000
Columns: 44
One row = one content item / page, with trailing-90-day performance signals.
          content_id         client_id  search_volume  competition competition_level  cpc    content_type   main_intent  word_count  char_count provider_used             model_used  impressions_90d  clicks_90d  pageviews_90d  sessions_90d  users_90d  engaged_sessions_90d  ai_sessions_90d  scroll_events_90d  days_with_impressions  days_with_sessions  impressions_last_30d  clicks_last_30d  sessions_last_30d  impressions_prev_30d  clicks_prev_30d  sessions_prev_30d  content_age_days age_tier  age_tier_order  days_since_last_update freshness_tier word_count_tier char_count_tier  ctr  avg_position  engagement_rate  scroll_rate  ai_traffic_pct impression_tier position_tier trend_direction  trend_pct
content_304f48230142 client_f369cb89fc           10.0         0.67              HIGH 2.05 keyword article transactional      3221.0     20457.0           NaN       gemini-2.5-flash             38

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The pattern is too messy because page decline is driven by multiple signals interacting at once: recent traffic direction, engagement, position, content type, and search demand. A one-line rule may catch some obvious cases, but real pages vary in seasonality, content age, and search behavior. A model is useful because it can learn a weighted combination of these noisy, nonlinear signals instead of relying on a brittle hand-coded rule.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from pathlib import Path
import pandas as pd

search_roots = [Path.cwd(), *Path.cwd().parents]

def find_dataset():
    for root in search_roots:
        candidate = root / "data" / "raw" / "content_refresh_anonymized.csv"
        if candidate.exists():
            return candidate
    for root in search_roots:
        matches = list(root.rglob("content_refresh_anonymized.csv"))
        if matches:
            return matches[0]
    return None

candidate = find_dataset()
if candidate is None:
    raise FileNotFoundError("Starter dataset not found under data/raw/")

df = pd.read_csv(candidate)
example_signals = [
    "impressions_90d",
    "engagement_rate",
    "ctr",
    "avg_position",
    "content_type",
    "word_count",
    "trend_pct"
]
print("The pattern blends many signals:")
print(example_signals)
print("This is why a simple if-statement is not enough: signals are noisy, overlapping, and different pages need different combinations.")


The pattern blends many signals:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used']
This is why a simple if-statement is not enough: signals are noisy, overlapping, and different pages need different combinations.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.